# Notebook 2 — Radio Quality KPI Forecasting
**Target KPI:** `bler_percent` (Block Error Rate)

| Model | Why |
|---|---|
| **LSTM** | Baseline — strong sequential learner |
| **BiLSTM** | Reads sequence forward + backward — catches error-rate patterns that build across multiple steps |
| **GRU** | Lighter alternative — well-suited for bursty radio signals |

In [1]:
import math
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import joblib
import os
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
# ─────────────────────────────────────────────
#  CONFIGURATION  —  edit these values freely
# ─────────────────────────────────────────────

DATA_PATH   = "../Data/Model_data.csv"
OUTPUT_DIR  = "../outputs/radio_quality"

TARGET_KPI  = "bler_percent"
TOLERANCE   = 0.5       # ± 0.5 % BLER

SEQ_LEN     = 12
TRAIN_RATIO = 0.8

HIDDEN_SIZE = 64
NUM_LAYERS  = 2
EPOCHS      = 10
BATCH_SIZE  = 2048
LR          = 1e-3

RNG_SEED = 42
DEVICE   = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
np.random.seed(RNG_SEED)
torch.manual_seed(RNG_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RNG_SEED)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cpu


In [4]:
print("Loading data...")
df = pd.read_csv(DATA_PATH, parse_dates=["timestamp"])
df = df.sort_values("timestamp").reset_index(drop=True)
print(f"Rows: {len(df):,} | Columns: {len(df.columns)}")
df.head(3)

Loading data...
Rows: 210,528 | Columns: 20


,timestamp,slice_type,latitude,longitude,one_way_latency_ms,jitter_ms,rtt_ms,packet_delay_budget_ms,handover_interruption_time_ms,reliability_percent,packet_loss_percent,packet_loss_rate_percent,bler_percent,throughput_dl_mbps,throughput_ul_mbps,spectral_efficiency_bps_hz,handover_success_rate_percent,energy_efficiency_bits_per_joule,anomaly,anomaly_type
0,2024-01-01 00:00:00,URLLC,33.800386,-7.547638,2.5865,0.5029,5.3423,0.7614,5.2166,99.9995,0.0005,0.0005,0.4930,106.5463,103.6793,9.9301,99.5036,476587788.0,0,normal
1,2024-01-01 00:05:00,URLLC,33.802700,-7.553952,2.4543,0.4950,5.1841,0.7626,5.0939,99.9995,0.0005,0.0005,0.4954,102.3002,102.2863,9.9559,99.4860,485576369.0,0,normal
2,2024-01-01 00:10:00,URLLC,33.800517,-7.556512,2.4245,0.4927,5.1083,0.7753,5.1232,99.9995,0.0005,0.0005,0.4994,97.0391,98.8266,9.9911,99.4985,490452024.0,0,normal


In [5]:
# Radio-layer features + lag columns
FEATURE_COLS = [
    "bler_percent",
    "reliability_percent",
    "packet_loss_percent",
    "packet_loss_rate_percent",
    "throughput_dl_mbps",
    "spectral_efficiency_bps_hz",
    "one_way_latency_ms",
]

df = df.dropna(subset=FEATURE_COLS).reset_index(drop=True)
print(f"Clean rows: {len(df):,}")

Clean rows: 210,528


In [6]:
print("\nBuilding sequences...")

X_raw = df[FEATURE_COLS].values
y_raw = df[TARGET_KPI].shift(-1).values
X_raw, y_raw = X_raw[:-1], y_raw[:-1]

X_all, y_all = [], []
for i in range(SEQ_LEN, len(X_raw)):
    X_all.append(X_raw[i - SEQ_LEN : i, :])
    y_all.append(y_raw[i])

X_seq = np.stack(X_all).astype(np.float32)
y_seq = np.array(y_all, dtype=np.float32).reshape(-1, 1)
print(f"Sequences: {len(X_seq):,} | X: {X_seq.shape} | y: {y_seq.shape}")


Building sequences...
Sequences: 210,515 | X: (210515, 12, 7) | y: (210515, 1)


In [7]:
train_n = int(len(X_seq) * TRAIN_RATIO)
X_train, X_test = X_seq[:train_n], X_seq[train_n:]
y_train, y_test = y_seq[:train_n], y_seq[train_n:]

scaler_X = StandardScaler()
X_train_scaled = scaler_X.fit_transform(X_train.reshape(-1, X_train.shape[2])).reshape(X_train.shape)
X_test_scaled  = scaler_X.transform(X_test.reshape(-1, X_test.shape[2])).reshape(X_test.shape)

scaler_y = StandardScaler()
y_train_scaled = scaler_y.fit_transform(y_train)
y_test_scaled  = scaler_y.transform(y_test)

X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32, device=DEVICE)
X_test_t  = torch.tensor(X_test_scaled,  dtype=torch.float32, device=DEVICE)
y_train_t = torch.tensor(y_train_scaled, dtype=torch.float32, device=DEVICE)
y_test_t  = torch.tensor(y_test_scaled,  dtype=torch.float32, device=DEVICE)

print(f"Train: {len(X_train_t):,} | Test: {len(X_test_t):,} | Features: {X_train_t.shape[2]} | Device: {DEVICE}")

Train: 168,412 | Test: 42,103 | Features: 7 | Device: cpu


In [8]:
# ── MODEL 1: LSTM ──────────────────────────────────────────────────────────────
class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# ── MODEL 2: BiLSTM ────────────────────────────────────────────────────────────
class BiLSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True,
                            bidirectional=True, dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size * 2, 1)   # *2 for bidirectional
    def forward(self, x):
        out, _ = self.lstm(x)
        return self.fc(out[:, -1, :])


# ── MODEL 3: GRU ───────────────────────────────────────────────────────────────
class GRUModel(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(input_size, hidden_size, num_layers, batch_first=True,
                          dropout=dropout if num_layers > 1 else 0.0)
        self.fc = nn.Linear(hidden_size, 1)
    def forward(self, x):
        out, _ = self.gru(x)
        return self.fc(out[:, -1, :])

In [9]:
def get_batches(X, y, batch_size):
    for i in range(0, len(X), batch_size):
        yield X[i : i + batch_size], y[i : i + batch_size]


def train_and_evaluate(model, name):
    print(f"\n{'─'*55}\n  Training: {name}\n{'─'*55}")
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)
    loss_fn   = nn.L1Loss()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        train_losses = []
        for bx, by in get_batches(X_train_t, y_train_t, BATCH_SIZE):
            optimizer.zero_grad()
            loss = loss_fn(model(bx), by)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        with torch.no_grad():
            preds = scaler_y.inverse_transform(model(X_test_t).cpu().numpy())
            true  = scaler_y.inverse_transform(y_test_t.cpu().numpy())
            mae   = mean_absolute_error(true, preds)
            rmse  = math.sqrt(mean_squared_error(true, preds))

        mean_train = np.mean(train_losses)
        scheduler.step(mean_train)
        print(f"  Epoch {epoch}/{EPOCHS} | Train MAE: {mean_train:.4f} | Test MAE: {mae:.4f} | RMSE: {rmse:.4f} | LR: {optimizer.param_groups[0]['lr']:.6f}")

    acc = np.mean(np.abs(true - preds) <= TOLERANCE) * 100
    return {"name": name, "model": model, "MAE": mae, "RMSE": rmse, "Accuracy": acc, "preds": preds, "true": true}

In [10]:
input_size = X_train_t.shape[2]
results = []
results.append(train_and_evaluate(LSTMModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE),   "LSTM"))
results.append(train_and_evaluate(BiLSTMModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE), "BiLSTM"))
results.append(train_and_evaluate(GRUModel(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE),    "GRU"))


───────────────────────────────────────────────────────
  Training: LSTM
───────────────────────────────────────────────────────
  Epoch 1/10 | Train MAE: 0.1001 | Test MAE: 0.0626 | RMSE: 0.6068 | LR: 0.001000
  Epoch 2/10 | Train MAE: 0.0859 | Test MAE: 0.0558 | RMSE: 0.5372 | LR: 0.001000
  Epoch 3/10 | Train MAE: 0.0774 | Test MAE: 0.0513 | RMSE: 0.4889 | LR: 0.001000
  Epoch 4/10 | Train MAE: 0.0725 | Test MAE: 0.0488 | RMSE: 0.4600 | LR: 0.001000
  Epoch 5/10 | Train MAE: 0.0693 | Test MAE: 0.0473 | RMSE: 0.4379 | LR: 0.001000
  Epoch 6/10 | Train MAE: 0.0673 | Test MAE: 0.0463 | RMSE: 0.4242 | LR: 0.001000
  Epoch 7/10 | Train MAE: 0.0657 | Test MAE: 0.0458 | RMSE: 0.4134 | LR: 0.001000
  Epoch 8/10 | Train MAE: 0.0645 | Test MAE: 0.0451 | RMSE: 0.4040 | LR: 0.001000
  Epoch 9/10 | Train MAE: 0.0636 | Test MAE: 0.0449 | RMSE: 0.3949 | LR: 0.001000
  Epoch 10/10 | Train MAE: 0.0624 | Test MAE: 0.0439 | RMSE: 0.3870 | LR: 0.001000

────────────────────────────────────────────────

In [11]:
print("\n" + "═"*62)
print(f"  MODEL COMPARISON  —  Target: {TARGET_KPI}")
print("═"*62)
print(f"  {'Model':<10} {'MAE':>10} {'RMSE':>10} {'Acc (±{:.1f})'.format(TOLERANCE):>14}")
print("  " + "─"*56)
best = min(results, key=lambda r: r["MAE"])
for r in results:
    marker = " ← BEST" if r["name"] == best["name"] else ""
    print(f"  {r['name']:<10} {r['MAE']:>10.4f} {r['RMSE']:>10.4f} {r['Accuracy']:>13.2f}%{marker}")
print("═"*62)


══════════════════════════════════════════════════════════════
  MODEL COMPARISON  —  Target: bler_percent
══════════════════════════════════════════════════════════════
  Model             MAE       RMSE     Acc (±0.5)
  ────────────────────────────────────────────────────────
  LSTM           0.0439     0.3870         99.16%
  BiLSTM         0.0406     0.3508         99.16% ← BEST
  GRU            0.0467     0.4250         99.13%
══════════════════════════════════════════════════════════════


In [12]:
# === PREDICTION TEST ===
# Test the best model on 4 sample rows from the test set
# Each row = a sequence of SEQ_LEN=12 steps -> predicts the next value of bler_percent

import pandas as pd
import torch
import numpy as np

sample_indices = [0, len(X_test)//3, 2*len(X_test)//3, len(X_test)-1]

best_model = best["model"]
best_model.eval()

rows = []
with torch.no_grad():
    for idx in sample_indices:
        seq_raw    = X_test[idx]
        seq_scaled = scaler_X.transform(seq_raw).astype(np.float32)
        seq_t      = torch.tensor(seq_scaled[np.newaxis, :, :], device=DEVICE)

        pred_scaled = best_model(seq_t).cpu().numpy()
        pred_val    = scaler_y.inverse_transform(pred_scaled)[0, 0]
        true_val    = float(y_test[idx, 0])
        error       = abs(pred_val - true_val)
        within      = "✓" if error <= TOLERANCE else "✗"

        rows.append({
            "Sample index":                                    idx,
            f"Last known {TARGET_KPI}":                     round(seq_raw[-1, FEATURE_COLS.index(TARGET_KPI)], 4),
            "Predicted (t+1)":                                round(pred_val, 4),
            "Actual (t+1)":                                   round(true_val, 4),
            "Abs Error":                                      round(error, 4),
            f"Within ±{TOLERANCE} %":                 "✓" if error <= TOLERANCE else "✗",
        })

df_test = pd.DataFrame(rows)
print(f"Best model  : {best['name']}")
print(f"Target KPI  : {TARGET_KPI}")
print(f"Tolerance   : ±{TOLERANCE} %")
print()
display(df_test)


Best model  : BiLSTM
Target KPI  : bler_percent
Tolerance   : ±0.5 %



,Sample index,Last known bler_percent,Predicted (t+1),Actual (t+1),Abs Error,Within ±0.5 %
0,0,0.5037,0.5009,0.5041,0.0032,✓
1,14034,0.5448,0.5270,0.5357,0.0087,✓
2,28068,5.1005,1.6094,1.7135,0.1041,✓
3,42102,0.4899,0.4928,0.4750,0.0178,✓
